# Machine Learning en Producción — Universidad ORT Uruguay

- Gabriela Carrasco - 259373
- Mateo Carballo - 343357
- Marcelo García - 350766


# EDA — Fall Detection

**Curso:** Machine Learning en Producción — ORT Uruguay

Clasificación binaria **Fall / Not Fall** con imágenes y metadatos tabulares derivados.

**Datasets (Roboflow):**
- DS1: `iot-c5yvo/fall-detection-raskl` v2
- DS2: `jay-buvdf/fa-nunl5` v1

| Clase | Descripción |
|-------|-------------|
| `fall` | Situación de caída |
| `not_fall` | Situación normal |

> Pipeline: `download_dataset.py` → `fuse_datasets.py` → `extract_features.py`

> Dependencias: `pip install -r requirements-dev.txt`

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

from src.preprocessing.eda_stats import (
    BINARY_CLASS_NAMES,
    FEATURE_COLS,
    build_metadata_dataframe,
    check_split_leakage_by_filename,
    check_train_test_leakage,
    collect_image_sizes,
    collect_source_balance,
    collect_split_stats,
    count_processed_classes,
    ensure_dataset_available,
    fused_dir,
    get_binary_class_weights,
    load_metadata,
    metadata_path,
    resolve_project_root,
    split_stats_to_dataframe,
)
from src.settings.config import settings

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

ROOT = resolve_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ensure_dataset_available(ROOT)
print('Proyecto:', ROOT)
print('Dataset:', fused_dir(ROOT))
print('Config image_size:', settings.image_size)

## 1. Distribución Fall / Not Fall por split

In [ ]:
df_splits = split_stats_to_dataframe(collect_split_stats(ROOT))
df_splits

In [ ]:
plot_df = df_splits.melt(
    id_vars=['split', 'total_images'],
    value_vars=['fall', 'not_fall'],
    var_name='clase', value_name='cantidad',
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=plot_df, x='split', y='cantidad', hue='clase', ax=ax)
ax.set_title('Distribución binaria por split')
plt.tight_layout()
plt.show()

total_fall = df_splits['fall'].sum()
total = df_splits['fall'].sum() + df_splits['not_fall'].sum()
print(f'Proporción global fall: {total_fall / total:.1%}')

## 2. Balance por fuente (DS1 vs DS2)

In [ ]:
df_source = collect_source_balance(ROOT)
if df_source.empty:
    print('Sin metadata de fuentes')
else:
    display(df_source)
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.barplot(data=df_source, x='split', y='count', hue='dataset_source', ax=ax)
    ax.set_title('Imágenes por split y dataset de origen')
    plt.tight_layout()
    plt.show()

## 3. Dimensiones de imágenes

In [ ]:
sizes = collect_image_sizes(ROOT)
df_sizes = pd.DataFrame(sizes, columns=['width', 'height'])
display(df_sizes.describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(df_sizes['width'], bins=15, ax=axes[0])
axes[0].set_title('Ancho (px)')
sns.histplot(df_sizes['height'], bins=15, ax=axes[1])
axes[1].set_title('Alto (px)')
plt.tight_layout()
plt.show()

print(f"Rango ancho: {df_sizes['width'].min()}–{df_sizes['width'].max()} px")

## 4. Features tabulares derivadas

In [ ]:
df_meta = load_metadata(ROOT)
has_features = all(c in df_meta.columns for c in FEATURE_COLS)
if not has_features:
    print('Features no encontradas. Ejecutar: python scripts/extract_features.py')
else:
    display(df_meta[FEATURE_COLS].describe().round(3))
    fig, ax = plt.subplots(figsize=(10, 4))
    corr = df_meta[FEATURE_COLS + ['label_code']].corr(numeric_only=True)
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
    ax.set_title('Correlación features ↔ label_code')
    plt.tight_layout()
    plt.show()

## 5. Muestras visuales

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, label in zip(axes.flatten(), ['fall', 'not_fall', 'fall', 'not_fall', 'fall', 'not_fall']):
    sample_dir = fused_dir(ROOT) / 'train' / label
    samples = sorted(sample_dir.glob('*.*'))[:1]
    if samples:
        img = Image.open(samples[0]).convert('RGB')
        ax.imshow(img)
        ax.set_title(f'{label}: {samples[0].name[:24]}', fontsize=8)
    ax.axis('off')
plt.suptitle('Ejemplos train — Fall vs Not Fall')
plt.tight_layout()
plt.show()

## 6. Pesos de clase y dataset procesado

In [ ]:
processed = count_processed_classes(ROOT)
df_processed = pd.DataFrame(processed).fillna(0).astype(int)
display(df_processed)

weights = get_binary_class_weights(ROOT)
print('Pesos (train):', dict(zip(BINARY_CLASS_NAMES, [round(w, 3) for w in weights])))

## 7. Metadata, leakage y exportación

In [ ]:
if df_meta.empty:
    df_meta = build_metadata_dataframe(ROOT)
print(f'Total registros: {len(df_meta)}')
display(df_meta.head())

out = metadata_path(ROOT)
out.parent.mkdir(parents=True, exist_ok=True)
df_meta.to_csv(out, index=False)
print(f'Metadatos guardados en: {out}')

In [ ]:
leakage_files = check_split_leakage_by_filename(ROOT)
if leakage_files:
    print('ADVERTENCIA: filenames duplicados entre splits')
    for issue in leakage_files:
        print(issue)
else:
    print('OK: no hay filenames duplicados entre splits')

base_leaks = check_train_test_leakage(ROOT)
if base_leaks:
    print('ADVERTENCIA: nombres base compartidos train/test:', base_leaks[:5])
else:
    print('OK: no hay leakage train/test por nombre base')

## 8. Conclusiones y decisiones del proyecto

In [ ]:
valid_n = int(df_splits.loc[df_splits['split'] == 'valid', 'total_images'].sum())
test_n = int(df_splits.loc[df_splits['split'] == 'test', 'total_images'].sum())
pct_fall = df_splits['fall'].sum() / (df_splits['fall'].sum() + df_splits['not_fall'].sum())
w_min, w_max = df_sizes['width'].min(), df_sizes['width'].max()

print('Hallazgos (calculados):')
print(f'  - Desbalance fall: {pct_fall:.1%}')
print(f'  - Valid: {valid_n} imgs | Test: {test_n} imgs')
print(f'  - Tamaños: {w_min}–{w_max} px → resize {settings.image_size}×{settings.image_size}')
print(f'  - Clases ImageFolder: {BINARY_CLASS_NAMES} (fall=0, not_fall=1)')

| Hallazgo | Decisión aplicada |
|----------|-------------------|
| Dos fuentes Roboflow (DS1+DS2) | Fusión con prefijos + metadata unificada |
| Desbalance Fall / Not Fall | `WeightedRandomSampler` + pesos en `CrossEntropyLoss` |
| Variabilidad de tamaño | Resize + normalización ImageNet en `transforms.py` |
| Test pequeño | Evaluación principal en **valid** |
| Features tabulares | `extract_features.py` + análisis en EDA |
| Riesgo de leakage | Chequeo por filename y nombre base train/test |

**Pipeline:**
1. `scripts/download_dataset.py`
2. `scripts/fuse_datasets.py`
3. `scripts/extract_features.py`
4. `src/preprocessing/dataset.py` + `src/core/train.py`
5. `python -m src.core.evaluate`